# Chapter 8 &mdash; Cross-Checking Two Designs by Minimal-DFA Isomorphism

**Concept 8 of the Chapter 8 decomposition:** *Cross-Checking Two Designs by Minimal-DFA Isomorphism*

`iso_dfa` on the two minimized machines returns True &mdash; the RE design and the NFA design agree.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8/Concept-Cross-Check-By-Isomorphism/Concept-Cross-Check-By-Isomorphism.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Concepts 6 and 7 built the **same language** two entirely different ways: a union of
dented patterns, and a layered error-correcting machine.

Minimize both and ask `iso_dfa`. By Myhill&ndash;Nerode a `True` answer means the two
designs denote **exactly** the same language &mdash; not "agree on the tests I thought
of".

This is the strongest routine check in the book, and it costs two function calls. Use
it whenever you have two independent designs; if it says `False`, `langeq_dfa` with
`gen_counterex=True` tells you where they part company.

## 2. Definitions

### Design 1: the RE with dents

In [ ]:
from itertools import combinations, product
TARGET, MAXD = '0101', 2
RE2 = '+'.join(''.join('(0+1)' if i in pos else TARGET[i] for i in range(4))
               for pos in combinations(range(4), 2))
D_re = min_dfa(nfa2dfa(re2nfa(RE2)))

### Design 2: the layered NFA

In [ ]:
def layered_nfa(target, maxd):
    lines, n = ['NFA'], len(target)
    def nm(i, d):
        if i == 0 and d == 0: return 'I'
        return ('F' if i == n else 'S') + '_p%d_d%d' % (i, d)
    for i, want in enumerate(target):
        other = '1' if want == '0' else '0'
        for d in range(maxd + 1):
            lines.append('%s : %s -> %s' % (nm(i, d), want, nm(i+1, d)))
            if d < maxd:
                lines.append('%s : %s -> %s' % (nm(i, d), other, nm(i+1, d+1)))
    return md2mc('\n'.join(lines))

D_nfa = min_dfa(nfa2dfa(layered_nfa(TARGET, MAXD)))

## 3. Tests

Both minimize to the same size.

In [ ]:
print("from the RE  : %d states" % len(D_re["Q"]))
print("from the NFA : %d states" % len(D_nfa["Q"]))
assert len(D_re["Q"]) == len(D_nfa["Q"])

**And they are isomorphic** &mdash; the two designs are the same language.

In [ ]:
print("langeq_dfa :", langeq_dfa(D_re, D_nfa))
print("iso_dfa    :", iso_dfa(D_re, D_nfa))
assert langeq_dfa(D_re, D_nfa) and iso_dfa(D_re, D_nfa)

Both match the Hamming specification independently.

In [ ]:
def ham(a, b): return sum(x != y for x, y in zip(a, b))
spec = lambda s: len(s) == 4 and ham(s, TARGET) <= 2
strs = [''.join(p) for k in range(7) for p in product('01', repeat=k)]
for name, D in [('RE design', D_re), ('NFA design', D_nfa)]:
    assert all(accepts_dfa(D, s) == spec(s) for s in strs)
    print("%-11s matches the spec on all %d strings" % (name, len(strs)))

If they had disagreed, `gen_counterex` would show where.

In [ ]:
wrong = min_dfa(nfa2dfa(layered_nfa(TARGET, 1)))     # distance 1 only
print("distance-1 design vs distance-2 design:")
print("  langeq :", langeq_dfa(D_re, wrong))
assert not langeq_dfa(D_re, wrong)
langeq_dfa(D_re, wrong, gen_counterex=True)

## 4. Animation

The one minimal machine that both designs produce.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(D_re, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Deliberately break one design and read the counterexample. Is it the shortest?
2. Why is `iso_dfa` on minimized machines stronger than any finite test suite?
3. Build a third design (a hand-drawn DFA) and check all three agree.

In [ ]:
# Your work for the exercises above.